# Module 4

## Prerequisites

In [1]:
import boto3
import json
import time
from datetime import datetime
import os

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

## Inference profile

In [2]:
prompt = """
What design patterns are used in this code?

You are an expert Python developer. Here's a codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

### Invoking cross-region inference profile

In [ ]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "6555cf76-5a07-424e-a9aa-a9aee9939484",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 30 Jun 2026 19:33:45 GMT",
      "content-type": "application/json",
      "content-length": "5688",
      "connection": "keep-alive",
      "x-amzn-requestid": "6555cf76-5a07-424e-a9aa-a9aee9939484"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Based on the provided code snippet, the `DataProcessor` class doesn't explicitly exhibit any well-known design patterns. However, there are some characteristics and potential patterns that can be inferred from the given context. Here are some possibilities:\n\n1. **Builder Pattern**:\n   The class takes a `config` object in its constructor. This could be indicative of the Builder pattern, where a complex object is built step-by-step with different configurations. The `config` object might encapsulate va

### Use your own inference profile

In [4]:
# create an inference profile
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

response = bedrock_control.create_inference_profile(
    inferenceProfileName='my-nova-lite-profile',
    description='Custom inference profile for Nova Lite',
    modelSource={
        'copyFrom': 'arn:aws:bedrock:us-east-1:767397992380:inference-profile/us.amazon.nova-lite-v1:0'
    }
)
inference_profile_arn = response['inferenceProfileArn']

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "785c0053-9e79-41c3-87dd-0cc49c85c604",
    "HTTPStatusCode": 201,
    "HTTPHeaders": {
      "date": "Tue, 30 Jun 2026 19:34:51 GMT",
      "content-type": "application/json",
      "content-length": "125",
      "connection": "keep-alive",
      "x-amzn-requestid": "785c0053-9e79-41c3-87dd-0cc49c85c604"
    },
    "RetryAttempts": 0
  },
  "inferenceProfileArn": "arn:aws:bedrock:us-east-1:767397992380:application-inference-profile/o0xbixfgd531",
  "status": "ACTIVE"
}


In [5]:
response = bedrock.converse(
    modelId=inference_profile_arn,
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "14f0b21c-69d2-4580-a216-9b01b47f2c66",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 30 Jun 2026 19:36:06 GMT",
      "content-type": "application/json",
      "content-length": "5031",
      "connection": "keep-alive",
      "x-amzn-requestid": "14f0b21c-69d2-4580-a216-9b01b47f2c66"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Based on the provided code snippet for the `DataProcessor` class, we can identify and discuss several design patterns that might be employed in this context, as well as patterns that could be considered for future implementation.\n\n### Design Patterns in the Provided Code\n\n1. **Builder Pattern**:\n   - The `DataProcessor` class uses an initializer (`__init__`) to set up the configuration and initialize an empty list for data. This pattern is useful when constructing complex objects step by step, and 

In [6]:
response = bedrock_control.delete_inference_profile(
    inferenceProfileIdentifier=inference_profile_arn
)

## Batch Inference

Upload jsonl file to S3

In [7]:
bucket_name = "batch-data-inference-dk01"

s3 = boto3.client('s3')

# Upload manifest to S3
input_key = 'input/city-reviews-manifest.jsonl'
s3.upload_file('input/city-reviews-manifest.jsonl', bucket_name, input_key)

Call the CreateModelInvocationJob API

In [8]:
accountnumber = "767397992380"

# Create the bedrock control plane client
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

# Create batch inference job
job_name = f"batch-job-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

response = bedrock_control.create_model_invocation_job(
    jobName=job_name,
    roleArn=f"arn:aws:iam::{accountnumber}:role/BedrockBatchRole",
    modelId="us.amazon.nova-lite-v1:0",
    inputDataConfig={
        's3InputDataConfig': {
            's3Uri': f's3://{bucket_name}/input/'
        }
    },
    outputDataConfig={
        's3OutputDataConfig': {
            's3Uri': f's3://{bucket_name}/output/'
        }
    }
)

job_arn = response['jobArn']


Get the Job Status

In [9]:
status_response = bedrock_control.get_model_invocation_job(jobIdentifier=job_arn)
status = status_response['status']
print(f"Job status: {status}")

Job status: Submitted


Instead of waiting, let's use a jobArn for a job we know already finished

In [10]:
completed_job_arn = job_arn

Wait for the job to finish

In [11]:
# Wait for job completion
while True:
    status_response = bedrock_control.get_model_invocation_job(jobIdentifier=completed_job_arn)
    status = status_response['status']
    print(f"Job status: {status}")
    
    if status in ['Completed', 'PartiallyCompleted', 'Failed', 'Stopped', 'Expired']:
        break
    
    time.sleep(30)



Job status: Submitted
Job status: Submitted
Job status: Submitted
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Scheduled
Job status: Scheduled
Job status: Scheduled
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: Completed


Download the result

In [12]:
if status in ["Completed", "PartiallyCompleted"]:
    # Download results
    output_objects = s3.list_objects_v2(Bucket=bucket_name, Prefix='output/')
    
    os.makedirs('batch-output', exist_ok=True)
    
    for obj in output_objects.get('Contents', []):
        if obj['Key'].endswith('/'):
            continue
            
        local_file = os.path.join('batch-output', obj['Key'].split('/')[-1])
        s3.download_file(bucket_name, obj['Key'], local_file)
        print(f"Downloaded: {local_file}")


Downloaded: batch-output\city-reviews-manifest.jsonl.out
Downloaded: batch-output\manifest.json.out


Stop the job we created to save on cost

In [ ]:
response = bedrock_control.stop_model_invocation_job(
    jobIdentifier=job_arn
)

## Prompt Caching with Nova Lite

In [13]:
# Large context that we want to cache
cached_context = """
You are a expert Python developer. Here's a large codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

First request that establishes the cache

In [14]:
firstRequest = bedrock.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default"
                    }
                },
                {
                    "text": "What design patterns are used in this code?"
                }
          ]
      }
    ]
)

print(json.dumps(firstRequest, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "77585212-b676-4a68-add5-cfe7cee99726",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 30 Jun 2026 19:55:35 GMT",
      "content-type": "application/json",
      "content-length": "3122",
      "connection": "keep-alive",
      "x-amzn-requestid": "77585212-b676-4a68-add5-cfe7cee99726"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Given the provided code snippet, there are several design patterns that can be identified. Let's analyze them in the context of the `DataProcessor` class:\n\n### 1. **Factory Method Pattern**\nThe `__init__` method in `DataProcessor` takes a `config` parameter. This suggests that there might be a factory method or a similar pattern used to create instances of `DataProcessor` with the appropriate configuration. \n\n### 2. **Template Method Pattern**\nThe methods `load_data`, `transform_data`, and `valida

Second request that uses cached context

In [15]:
secondRequest = bedrock.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default"
                    }
                },
                {
                    "text": "what else can you tell me about this code?"
                }
          ]
      }
    ]
)

print(json.dumps(secondRequest, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "63d41e00-466a-4e66-9329-3abd9992176a",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 30 Jun 2026 19:55:54 GMT",
      "content-type": "application/json",
      "content-length": "5836",
      "connection": "keep-alive",
      "x-amzn-requestid": "63d41e00-466a-4e66-9329-3abd9992176a"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Based on the provided code snippet and the context that this class is part of a larger system with many similar classes, there are several observations and recommendations I can provide:\n\n### Observations:\n\n1. **Initialization and Configuration:**\n   - The `__init__` method initializes the `config` and an empty list `data`.\n   - This suggests that `config` is a crucial part of the class's operation, possibly containing settings or parameters needed for loading, transforming, and validating data.\n

Compare the usage

In [16]:
print("### First request ###")
print("Usage:", json.dumps(firstRequest['usage'], indent=2))
print("Metrics:", json.dumps(firstRequest['metrics'], indent=2))

print("\n\n### Second request ###")
print("Usage:", json.dumps(secondRequest['usage'], indent=2))
print("Metrics:", json.dumps(secondRequest['metrics'], indent=2))

### First request ###
Usage: {
  "inputTokens": 9,
  "outputTokens": 573,
  "totalTokens": 706,
  "cacheReadInputTokens": 0,
  "cacheWriteInputTokens": 124
}
Metrics: {
  "latencyMs": 4717
}


### Second request ###
Usage: {
  "inputTokens": 10,
  "outputTokens": 1363,
  "totalTokens": 1497,
  "cacheReadInputTokens": 124,
  "cacheWriteInputTokens": 0
}
Metrics: {
  "latencyMs": 8726
}


## Converse on documents
### Using `bytes` as a source

In [17]:
with open('input/AnyCompany_financial_10K.pdf', "rb") as file:
    doc_bytes = file.read()

In [18]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "document": {
                        "format": "pdf",
                        "name": "AnyCompany_financial",
                        "source": {
                            "bytes": doc_bytes
                        }
                    }
                },
                {
                    "text": "What investments have AnyCompany made?"
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "79a7c1ec-47c0-45c5-ad5c-502f94a14d41",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 30 Jun 2026 19:57:40 GMT",
      "content-type": "application/json",
      "content-length": "317",
      "connection": "keep-alive",
      "x-amzn-requestid": "79a7c1ec-47c0-45c5-ad5c-502f94a14d41"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "AnyCompany has made significant investment positions in companies A, B, and C, amounting to $1.2 billion."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 194688,
    "outputTokens": 25,
    "totalTokens": 194713
  },
  "metrics": {
    "latencyMs": 48528
  }
}


### Using s3Location as a source

In [19]:
bucket_name = "batch-data-inference-dk01"

s3 = boto3.client('s3')

input_key = 'AnyCompany_financial_10K.pdf'
s3.upload_file('input/AnyCompany_financial_10K.pdf', bucket_name, input_key)

## Conversation history

### Create the DynamoDB Table and set the time to live

In [20]:
table_name = "conversation-history"

dynamodb = boto3.client('dynamodb', region_name="us-east-1")
response = dynamodb.create_table(
    TableName=table_name,
    BillingMode='PAY_PER_REQUEST',
    AttributeDefinitions=[
        {
            'AttributeName': 'userId',
            'AttributeType': 'S'
        },
        {
            'AttributeName': 'timestamp',
            'AttributeType': 'N'
        }
    ],
    KeySchema=[
        {
            'AttributeName': 'userId',
            'KeyType': 'HASH'
        },
        {
            'AttributeName': 'timestamp',
            'KeyType': 'RANGE'
        }
    ]
)

# Wait for table to be active
waiter = dynamodb.get_waiter('table_exists')
waiter.wait(TableName=table_name)

ttl_response = dynamodb.update_time_to_live(
    TableName=table_name,
    TimeToLiveSpecification={
        'AttributeName': 'ttl',
        'Enabled': True
    }
)

### Create the functions to get the conversation history and store new messages

In [21]:
table_name = "conversation-history"
ddbclient = boto3.client('dynamodb', region_name = "us-east-1")

def get_conversation_history(user_id):
    # Pagination is required in case the conversation is more than 1MB.
    paginator = ddbclient.get_paginator('query')
    pages = paginator.paginate(
        TableName = table_name,
        KeyConditionExpression = 'userId = :val',
        ExpressionAttributeValues = {':val': {'S': user_id}}
    )
    messages = []
    for page in pages:
        for item in page.get('Items', []):
            messages.append({
                "role": item["role"]["S"],
                "content": [{ "text": item["message"]["S"] }]
            })
    return messages

In [22]:
dynamodbResource = boto3.resource('dynamodb', region_name = "us-east-1")
conversation_table = dynamodbResource.Table(table_name)

def store_message(user_id, message, role):
    now_in_seconds = int(time.time())
    expire_ttl = now_in_seconds + (1 * 24 * 60 * 60) # 1 day
    
    conversation_table.put_item(
        Item = {
            'userId': user_id,
            'timestamp': now_in_seconds,
            'message': message,
            'role': role,
            'ttl': expire_ttl
        }
    )


### Create the function to send messages to the LLM

In [23]:
def send_message(user_id, message):
    
    # Load the conversation history
    conversation_messages = get_conversation_history(user_id)

    # Store the new message
    store_message(user_id, message, "user")
    
    # Add the new message to the conversation
    conversation_messages.append({
        "role": "user",
        "content": [{ "text": message }]
    })

    # Send the conversation to the LLM
    model_response = bedrock.converse(
        modelId="amazon.nova-lite-v1:0",
        messages=conversation_messages,
        system = [{ 
            "text": "Please provide a helpful, conversational response based on the available information and conversation history." 
        }],
        inferenceConfig = {
            "maxTokens": 300,
            "temperature": 0.7,
            "topP": 0.9
        }
    )

    # Parse the message from the LLM, store it and return it
    assistant_msg = model_response['output']['message']['content'][0]['text']
    store_message(user_id, assistant_msg, "assistant")

    return conversation_messages, assistant_msg

### Sending messages

In [24]:
user_id = "Disha"
convo, response = send_message(user_id, "My name is Disha and I love to eat pizza. What are some good pizza places near me in Vancouver?")
print(response)


Hi Disha! Vancouver has a fantastic pizza scene with many options to suit different tastes. Here are some highly recommended pizza places you might want to check out:

1. **L'Antica Pizzeria da Michele** (Granville Island)
   - Famous for its Neapolitan-style pizza, it's a favorite among locals and tourists alike.

2. **Joe Fortes Pizza Co.** (Gastown and other locations)
   - Known for its delicious wood-fired pizzas, it offers a variety of toppings and has a cozy atmosphere.

3. **Lou's Pizza** (Mount Pleasant)
   - Offers classic pizza with a modern twist, and their sourdough crust is a standout.

4. **Pizzeria Libretto** (Mount Pleasant)
   - This spot is well-loved for its creative toppings and unique combinations.

5. **Via Pizzeria** (Kitsilano)
   - Offers a range of pizzas with a focus on fresh, high-quality ingredients.

6. **Supremo Pizzeria** (Commercial Drive)
   - Known for its unique toppings and gourmet pizzas, this place is a hidden gem.

7. **La Paloma** (Commercial D

In [25]:
convo, response = send_message(user_id, "What do I love to eat?")
print(response)

It sounds like you have a particular fondness for pizza, Disha! It’s a delicious and versatile food that’s enjoyed by many people around the world. If you ever want to explore more food options or have questions about different cuisines, feel free to ask. Enjoy your pizza adventures in Vancouver!


In [27]:
print(json.dumps(convo, indent=2))

[
  {
    "role": "user",
    "content": [
      {
        "text": "My name is Disha and I love to eat pizza. What are some good pizza places near me in Vancouver?"
      }
    ]
  },
  {
    "role": "assistant",
    "content": [
      {
        "text": "Hi Disha! Vancouver has a fantastic pizza scene with many options to suit different tastes. Here are some highly recommended pizza places you might want to check out:\n\n1. **L'Antica Pizzeria da Michele** (Granville Island)\n   - Famous for its Neapolitan-style pizza, it's a favorite among locals and tourists alike.\n\n2. **Joe Fortes Pizza Co.** (Gastown and other locations)\n   - Known for its delicious wood-fired pizzas, it offers a variety of toppings and has a cozy atmosphere.\n\n3. **Lou's Pizza** (Mount Pleasant)\n   - Offers classic pizza with a modern twist, and their sourdough crust is a standout.\n\n4. **Pizzeria Libretto** (Mount Pleasant)\n   - This spot is well-loved for its creative toppings and unique combinations.\n\n